# TODO
* Fixa en alternativ version av varje LLM så att istället för att bara prompta om att svara på frågan ska den nu prompta för att
  göra en tidigare text bättre. För varje generation efter den första ska den ta system prompt, bästa draften (som den tidigare genererat), delta feedback som beskriver vad som ändrats sista rundan,nya problem, färdiga problem och kvarvarande problem, och sista är user prompt.

* Bygg upp en improvement-loop och testa så att det blir bättre
* I körloopen så bygg upp så allting läggs i en JSON fil, varje sample etc. för att om det skulle krasha så slipper man köra om
* Gör KL-Divergence. Men istället för att försöka hitta en bok som beskriver och jämföra, så generera svaret utan några begränsningar alls och jämför distribution med den, vilket gör att man kan läsa av stylistiska skillnader mellan en vanlig välfungerande generation och vad som skiljer sig i denna.
* Bygg upp Lexical constraints graf. Gör så att x axeln är varje improvement loop, y axeln är hur mycket error från target 90% och averagea och sen kör standard deviation. Sen gör också en procent som visar hur många samples som successfully nådde sig inom gränsen.
* Kör varje modell för:
  - 20/50/100 i sample size *
  - 3-10 i improvement loop *
  - 3-10 st olika batch sizes (100, 500, 2000 kanske?) *
  - För alla 3 språk *
Bygg en convergence här så om alla measurements har stabiliserat, sluta i förväg.
* Presentera resultat över 100 samples (alt. 50 om för dyrt).
* Skriv rapporten

# NILAS (Nuimio Intelligent Language Assistant System)

NILAS (Nuimio Intelligence Language Agent System) is a subsystem developed for the award-winning language platform [Nuimio](https://nuimio.com). The project aims to enable seamless integration of AI & Machine Learning into the current algorithmic nature of Nuimio. It uses a flexible architecture built on iterative improvements using a varied set of closed-source models for increased performance.

This is the main file for Dennis Mitzeus' bachelor thesis project for the program Applied Artificial Intelligence 2026 at Halmstad University. The project includes data preprocessing, project structuring, technical implementation, and evaluation.

The current Architecture looks like this:

<img width="400px" src="./figures/artifact_flowchart_new.png"></img>

## Structure

This project will go through

1. Data Preprocessing
2. API setup
3. Technical Implementation
4. Evaluation

## Project

In [1]:
# Setting up warning and logging filters, and mutliprocessing before anything else.
import os
import warnings
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # only show actual errors
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

import logging
logging.getLogger("vllm").setLevel(logging.ERROR)
logging.getLogger("vllm.engine").setLevel(logging.ERROR)

import torch.multiprocessing as mp
from vllm import LLM, SamplingParams

mp.set_start_method("spawn", force=True)

In [2]:
# Fixes need to restart kernel everytime I update components in separate files
%load_ext autoreload
%autoreload 2

%matplotlib inline

# Import Declarations
from IPython.display import Markdown, display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import seaborn as sns
import re
import contractions
import requests
import tarfile
import shutil
import csv
from dotenv import load_dotenv, dotenv_values
import os
import json
import spacy
import sqlite3
import time

In [ ]:
TOTAL_SIZE = 2000 # Total size of each flashcard set
BATCH_SIZE = 20 # How many words get introduced at a time eg. 20: ask LLM with 20 flashcards, then again with 40, then 60,..., until total size reached
MODEL_SAMPLE_SIZE = 20 # How many times to ask the same question (to get a distribution of probabilistic results)

### Data Preprocessing

#### Status for language Datasets Found:
* ✅ Swedish: Kelly (https://spraakbanken.gu.se/resurser/kelly)
* ✅ Spanish: ChatSubs (https://linkinghub.elsevier.com/retrieve/pii/S2352340923006650) (Data link: https://github.com/conversa-ai/ChatSubs)
* ✅ Korean: 한국어 학습용 어휘 목록(엑셀 파일) (TOPIK, from National Institute of Korean Language) (https://korean.go.kr/front/etcData/etcDataView.do?etc_seq=71)

#### Dataset explanations

##### Swedish

The Swedish dataset "Kelly" is used for Swedish. Swedish dataset contains 9 columns: "ID", "Raw freq", "WPM" "CEFR levels", "Source", "Grammar", "Swedish items for translation", "Word classes", and "Examples". 

Raw freq corresponds to the total frequency over the whole extracted corpus, WPM corresponds to a ratio of how many words per 1 000 000 words is this word, CEFR levels correspond to how advanced a word is following the Common European Framework of Reference for Languages, Source is from which corpus the word comes from, Grammar corresponds to certain prefixes or suffixes some word classes inherit (for example "att" before all root verbs), Swedish items for translation is the word itself, word classes correspond to which grammatical type a word is (more details under "Lookup" workspace in file datasets/1-raw/swedish.xls), lastly, examples provide example sentences of use for some words.

##### Spanish

The Spanish dataset "ChatSubs" is a collection of almost 20 000 000 dialogue subtitles from various kinds of movies and series. As this is not a preprocessed list of the most common words, it needs preprocessing.

By structuring all dialogues to be inputted directly into an NLP preprocessing library such as spaCy, a full database of lemmas, their PoS (grammar type) and frequency can be constructed. The frequency and PoS can then be used to determine which words in each grammar type qualify for the final flashcard set.

##### Korean

Korean dataset is from the National Korean Language Institute which is a put together list of the most common words in the Korean language for foreign learners practicing for the TOPIK (Test of Proficiency in Korean) test. Korean dataset contains 5 columns: "순위", "단어", "품사", "풀이", and "등급".

순위 corresponds to the frequency ranking based on a 2002 researcher report named "Survey on Modern Korean Usage Frequency", 단어 corresponds to the word itself, 품사 refers to the grammatical type a word is (more details can be found at [the official dataset site](https://korean.go.kr/front/etcData/etcDataView.do?etc_seq=71)),풀이 refers to the Hanja (漢字) versions of some words, and 등급 refers to which TOPIK level a word is in where A means basic B is intermediate and C is advanced level.

### Preprocessing based on some criteria

For our preprocessing we need to fulfill some critera to get a good subset of the datasets viable for testing in the thesis architecture. 

It requires:

1. Vocabulary list of 100 Words per Language
2. A varied set of grammar types (Nouns, Verbs etc.), some weight more than others such as verbs being more important than conjunctions for basic language.
3. The words should be the most common in most cases. This means relative frequency is better than raw frequency.

#### Batches

Batches are subsets of a final dataset for gradual introduction into the thesis architecture system and works similar to smaller independent datasets inside the bigger one, meaning each batch will have similar word type ratios and not purely based on commonality (in case of a vocabulary list of 100 words, most adjectives may be top 60 meaning no adjectives can ever be used until batches after top 60 gets introduced)

This is a visual image of batches: 

<img src="./figures/flashcard_batches.png" width="400px"></img>

### Preprocessing

In [ ]:
from src.preprocessing.language.general import create_sorted_flashcard_set

In [ ]:
# This is the final structure of each preprocessed final list
final_example_df = pd.DataFrame(
 columns=["word", "pos", "frequency"]
)

for i in range(TOTAL_SIZE):
    final_example_df = pd.concat([final_example_df, pd.DataFrame([{"word": "example word", "pos": "verb", "frequency": 1}])], ignore_index=True)

final_example_df

#### Swedish

In [ ]:
from src.preprocessing.language.swedish import remove_and_merge_pos as swedish_remove_and_merge_pos

In [ ]:
swedish_raw = pd.read_csv("data/2-extracted/swedish.csv", sep=";")  # import
swedish_raw.isnull().sum()

In [ ]:
# changing WPM to numeric
swedish_raw["WPM"] = swedish_raw["WPM"].str.replace(",", ".").astype("Float64")

# drop rows with super high WPM values (outliers)
swedish_raw = swedish_raw[swedish_raw["WPM"] < 1000000]

In [ ]:
swedish_clump_word_classes = {
    "noun": ["noun", "noun-en", "noun-ett", "noun-en/-ett"],
    "verb": ["verb", "aux verb"],
}

swedish_processed = swedish_remove_and_merge_pos(data=swedish_raw, clump_word_classes=swedish_clump_word_classes)

In [ ]:
swedish_final, class_prior_fig, hamilton_fig = create_sorted_flashcard_set(
    data=swedish_processed,
    data_columns=["Swedish items for translation", "Word classes", "WPM"],
    pos_str="pos",
    frequency_str="frequency",
    rank_by="frequency",
    lang="swedish",
    target_columns=["word", "pos", "frequency"],
    drop_pos=["numeral", "proper name", "particip"],
    limit=TOTAL_SIZE,
)

display(swedish_final)
display(class_prior_fig[0])
display(hamilton_fig[0])

#### Spanish

In [ ]:
from src.preprocessing.language.spanish import load_dataset as load_spanish_dataset
from src.preprocessing.language.spanish import (
    extract_data_from_dataset as spanish_extract_data_from_dataset,
)
from src.preprocessing.language.spanish import (
    grammar_preprocessing as spanish_grammar_preprocessing,
)
from src.preprocessing.language.spanish import (
    finalize_dataset as spanish_finalize_dataset,
)
from src.preprocessing.language.spanish import remove_artifact_entries as spanish_remove_artifact_entries

In [ ]:
load_spanish_dataset() # downloads Spanish Dataset (ChatSubs) to /data/1-raw/spanish/

In [ ]:
spanish_extract_data_from_dataset(limit=20000, limit_sampling_seed=42) # extracts all dialogues from all documents into one .csv document
# spanish_extract_data_from_dataset(limit=1) # extracts all dialogues from all documents into one .csv document

In [ ]:
spanish_df = spanish_grammar_preprocessing(
    nlp_size="small",
    cores_to_use=6,
    import_chunk_size=120000,
    processing_chunk_size=4000,
)  # Does preprocessing: tokenization, lemmatization, PoS tagging

print(spanish_df.sort_values(by="frequency", ascending=False).head(10))

In [ ]:
spanish_raw = spanish_finalize_dataset(limit=5000) # writes and gives top N entries
# spanish_raw = spanish_finalize_dataset(limit=TOTAL_SIZE) # writes and gives top N entries

spanish_raw

In [ ]:
spanish_processed = spanish_remove_artifact_entries(spanish_raw, word_column="lemma")

In [ ]:
spanish_final, class_prior_fig, hamilton_fig = create_sorted_flashcard_set(
    data=spanish_processed, 
    data_columns=["lemma", "pos", "WPM"], 
    pos_str="pos",
    frequency_str="frequency",
    rank_by="frequency",
    lang="spanish",
    target_columns=["word", "pos", "frequency"], 
    drop_pos=["PROPN", "NUM", "PUNCT"],
    limit=TOTAL_SIZE)

display(spanish_final)
display(class_prior_fig[0])
display(hamilton_fig[0])

#### Korean

In [ ]:
korean_raw = pd.read_csv("data/2-extracted/korean.csv", sep=";")  # import
korean_raw.isnull().sum()

In [ ]:
korean_raw

In [ ]:
korean_raw = korean_raw.dropna(subset=["순위"]) # drops words where there is no ranking

In [ ]:
korean_raw["품사"].unique()

In [ ]:
# remove 단어 numbers
korean_raw["단어"] = korean_raw["단어"].str.replace(
    r"\d+$", "", regex=True
)

In [ ]:
# replace korean names for display purposes
korean_raw["품사"] = korean_raw["품사"].replace(
    {
        "명": "noun",
        "동": "verb",
        "부": "adv",
        "형": "adj",
        "보": "aux",
        "의": "dep",
        "관": "det",
        "불": "neg",
        "대": "pron",
        "수": "num",
        "감": "intj",
    }
)

In [ ]:
# As TOPIK dont have any frequency ranking, I'll weight the ranking instead
korean_raw["frequency_est"] = 1 / korean_raw["순위"]

In [ ]:
korean_final, class_prior_fig, hamilton_fig = create_sorted_flashcard_set(
    data=korean_raw,
    data_columns=["단어", "품사", "frequency_est"],
    pos_str="pos",
    frequency_str="frequency",
    rank_by="frequency",
    lang="korean",
    target_columns=["word", "pos", "frequency"],
    drop_pos=[],
    limit=TOTAL_SIZE,
)

display(korean_final)
display(class_prior_fig[0])
display(hamilton_fig[0])

In [ ]:
# imports
# from openai import OpenAI
import openai as client

from ai.OLD_models import Conversation_Model, Conversation
from src.ai.prompts import REVISED_PROMPT_STRING as SYSTEM_PROMPT_STRING

In [ ]:
load_dotenv()
# client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
client.api_key = os.getenv("OPENAI_KEY")

In [ ]:
swedish_vocab = pd.read_csv("data/3-final/swedish100.csv").sort_values(by="frequency", ascending=False)
spanish_vocab = pd.read_csv("data/3-final/spanish100.csv").sort_values(
    by="frequency", ascending=False
)
korean_vocab = pd.read_csv("data/3-final/korean100.csv").sort_values(
    by="frequency", ascending=False
)

In [ ]:
test_questions_swedish = [  # Make some test questions
    "What's the difference between 'en' and 'ett' in swedish?",
    # "When do you use 'är' vs 'har' in swedish?",
    # "Why do you say 'jag heter' instead of 'jag är' in swedish?",
    # "What does 'lagom' mean in swedish?",
    # "How do I know if a word is a 'ett' or 'en' word in swedish?",
]

test_questions_spanish = []
test_questions_korean = []

In [ ]:
print(
    f"""
Current Setup:
    Total Nr. Questions Swedish: {len(test_questions_swedish)}
    Total Nr. Questions Spanish: {len(test_questions_spanish)}
    Total Nr. Questions Korean: {len(test_questions_korean)}

    Flashcard Set Size:  {TOTAL_SIZE}
    Batch Size:          {BATCH_SIZE}
    Total Nr. Batches:   {TOTAL_SIZE // BATCH_SIZE}

    Total Nr. Samples (of generator model):   {MODEL_SAMPLE_SIZE}
"""
)

In [ ]:
# TEST

conversation = Conversation_Model(
    SYSTEM_PROMPT_STRING,
    model_client=client,
    model_name="gpt-5.1",
    keep_history=True,
    save_history_to_file=False,
    max_output_tokens=10,
    tokenwise_generation=True,
)

conv = ""
last_complete_token = None
token_part = ""

conversation.import_word_library(swedish_vocab)

In [ ]:
fully_finished = False
while fully_finished == False:
    new_prompt = False if len(conv) > 0 else True


    response, response_top_logprobs, generated_response, is_finished = (
        conversation.generate_tokenwise(
            'What is the difference between "en" and "ett" in swedish? Explain in only one short sentence.',
            conv,
            n_tokens=1,
            top_logprobs=20,
            new_prompt=new_prompt
        )
    )

    conv += generated_response

    fully_finished = is_finished

    if generated_response[0] == " ":
        # hit a new sequence
        last_complete_token = token_part
        token_part = generated_response.replace(" ", "")
    else:
        token_part += generated_response

    print(f"Last completed token: {last_complete_token}")
    print(f"Part token: {token_part}")
    print(f"Is finished: {is_finished}")


In [ ]:
for part in conversation.history:
    print(f"{part["role"]}: {part["content"] if part["role"] == "assistant" else ""}")

In [ ]:
generated_response

In [ ]:
conv

In [ ]:
conversation = ""
conversations_data = []

total_runs = (
    (TOTAL_SIZE // BATCH_SIZE) * len(test_questions_swedish) * MODEL_SAMPLE_SIZE
)

print(f"A total of {total_runs} conversations will be generated.")

run_counter = 0
for question_i, question in enumerate(test_questions_swedish):
    for batch_i in range(1):
    # for batch_i in range(TOTAL_SIZE // BATCH_SIZE):
        for sample_i in range(MODEL_SAMPLE_SIZE):
            run_counter += 1

            vocab_subset = swedish_vocab[: (batch_i + 1) * BATCH_SIZE]

            conversation = Conversation_Model(
                SYSTEM_PROMPT_STRING,
                model_client=client,
                model_name="gpt-5.2",
                keep_history=True,
                save_history_to_file=False,
                max_output_tokens=10,
                tokenwise_generation=True,
            )

            # add flashcard limits
            conversation.import_word_library(vocab_subset)

            prompt = question
            response_obj, response = conversation.ask(prompt)
            print(
                f"{run_counter}/{total_runs}: Sample {sample_i + 1} of question nr {question_i + 1} using {(batch_i + 1) * BATCH_SIZE} words Generated."
            )
            # print(f"You: {question}")
            # print(f"Model: {response}")
            # print("\n")

            conversations_data.append(
                Conversation(
                    id=f"{batch_i + 1}"
                    + f"{(batch_i + 1) * BATCH_SIZE}"
                    + f"{sample_i + 1}",
                    question_id=question_i,
                    sample_id=sample_i,
                    nr_vocab=(batch_i + 1) * BATCH_SIZE,
                    question=question,
                    response=response,
                    word_limits=conversation.flashcards,
                )
            )

            # as JSON
            with open(
                f"chats/test_questionings/question_{question_i + 1}_batchsize_{(batch_i + 1) * BATCH_SIZE}_sample_{sample_i + 1}.json",
                "w",
            ) as f:
                json_history = json.dumps(conversation.history, indent=4)
                f.write(json_history)

print("Done!")

### Technical Implementation

Technical implementation involves prompting, model design, flashcard imports, batch managements and running to gather data about the performance of measurable components of the system.

In this case testing to ask about different components of the swedish language and see how it performs.

In [4]:
swedish_vocab = pd.read_csv("data/3-final/swedish100.csv").sort_values(
    by="frequency", ascending=False
)
spanish_vocab = pd.read_csv("data/3-final/spanish100.csv").sort_values(
    by="frequency", ascending=False
)
korean_vocab = pd.read_csv("data/3-final/korean100.csv").sort_values(
    by="frequency", ascending=False
)

In [5]:
test_questions_swedish = [  # Make some test questions
    "What's the difference between 'en' and 'ett' in swedish?",
    "When do you use 'är' vs 'har' in swedish?",
    "Why do you say 'jag heter' instead of 'jag är' in swedish?",
    "What does 'lagom' mean in swedish?",
    "How do I know if a word is a 'ett' or 'en' word in swedish?",
]

test_questions_spanish = [
    "What is the difference between 'el' and 'la' in spanish?"
    "What is the difference between 'está' and 'estaba' in spanish?"
    "" # TODO
]
test_questions_korean = [] # TODO

In [6]:
print(
    f"""
Current Setup:
    Total Nr. Questions Swedish: {len(test_questions_swedish)}
    Total Nr. Questions Spanish: {len(test_questions_spanish)}
    Total Nr. Questions Korean: {len(test_questions_korean)}

    Flashcard Set Size:  {TOTAL_SIZE}
    Batch Size:          {BATCH_SIZE}
    Total Nr. Batches:   {TOTAL_SIZE // BATCH_SIZE}

    Total Nr. Samples (of generator model):   {MODEL_SAMPLE_SIZE}
"""
)


Current Setup:
    Total Nr. Questions Swedish: 5
    Total Nr. Questions Spanish: 1
    Total Nr. Questions Korean: 0

    Flashcard Set Size:  2000
    Batch Size:          20
    Total Nr. Batches:   100

    Total Nr. Samples (of generator model):   10



I would think the easiest way is to do one call to analyze the target and preferred language of the user before answering, then i prompt the vLLM to answer in the preferred language, if preferred language and target is the same, then apply a penalty to words outside of the vocabulary list, then when extract top logprobs and base the token off of some custom logic. This is my proposal. As for beam search, it would be at word level, as you proposed in strategy B.

so you propose I could instaed of using logit_bias I do the inverse and apply that penalty on the top logprobs N+extra padding first and then pick the top N

For stopping the search:
I will explain now what I think you mean and you correct me if any intuition is incorrect. So I understood it as when a beam reaches an EOS tag, I have it and calculate a normalized probability off of it, then you close that beam and continue on the ones yet not completed, but now you calculate normalized probability for each beam in each step, if the probability already is worse than the completed sequence, prune it as it can never win and continue on the ones that still has a better probability, and replace the finished one with the next finished one if that one has the better normalized probability until all beams are worse than the frozen one, then that one had the highest normalized probability. Is this correct?

In [7]:
from src.ai.vLLM import vLLM_run, initialize_vLLM
from vllm.transformers_utils.tokenizer import get_tokenizer
from openai import OpenAI

In [ ]:
# HYPERPARAMETERS
# --------------------------------------------/>

# Models
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct-AWQ"     # Light model with AWQ quantization
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct-AWQ"   # Heavier model with AWQ quantization

# vLLM Initialization
DTYPE = "half"                                  # dtype
GPU_MEMORY_UTILIZATION = 0.6                    # GPU Memory Utilization
MAX_MODEL_LEN = 30000                           # Maximum allowed tokens to be generated by the vLLM model. 
ENABLE_PREFIX_CACHING = True                    # IMPORTANT: "caches the KV cache of existing queries, so that a new query can directly reuse the KV cache if it shares the same prefix with one of the existing queries, allowing the new query to skip the computation of the shared part." -- vLLM Documentation (https://docs.vllm.ai/en/latest/features/automatic_prefix_caching/)

# Sampling Parameters
TEMPERATURE = 0.5                               # Temperature
# TOP_K = 0.9                                   # Top K
MAX_TOKENS = 1                                  # Max generated tokens at a time. Keep at 1 for beam search logic
BEAM_SIZE = 2                                   # Beam Size
BEAM_CONSTRAINT_PADDING = 20                    # Extra padding to have backup logprobs in case too many gets pruned from vocabulary or probability constraints
LOGPROBS = BEAM_SIZE + BEAM_CONSTRAINT_PADDING  # Number of logprobs to extract from vLLM
REPETITION_PENALTY = 2                          # vLLM repetition penalty, default 1 (https://docs.vllm.ai/en/v0.6.5/dev/sampling_params.html)
PRESENCE_PENALTY = 0.0                          # vLLM presence penalty, default 0 (https://docs.vllm.ai/en/v0.6.5/dev/sampling_params.html)
FREQUENCY_PENALTY = 0.2                         # vLLM frequency penalty, default 0 (https://docs.vllm.ai/en/v0.6.5/dev/sampling_params.html)
LOGITS_PREPROCESSORS = []                       # vLLM Logits Preprocessors
LOGPROB_NORM_ALPHA = 0.2                        # Weight variable for Normalized Logprobs, 1 = fully equal, makes longer sequences always better, 0 = no effect, default = 0.6

# Sequence Generation
MAX_RESPONSE_TOKEN_LENGTH = 300                 # Nr allowed token length for the final sequence length
USE_WORD_CONSTRAINT = True                      # Applies penalty term on words' logprob which are not in allowed vocabulary
WORD_CONSTRAINT_TYPE = "soft"                   # "soft" | "hard", applies 
WORD_SOFT_CONSTRAINT_PENALTY = 1                # Soft penalty to discourage use of non-allowed words while allowing some slack. Increase if using too many non-allowed words

# Misc
VERBOSE = "sequence"                            # "full" | "sequence" | None Prints progress from inference

# --------------------------------------------/>

# Improvement Loop

# --------------------------------------------/>

In [9]:
# vLLM
llm = initialize_vLLM(
    model=MODEL_NAME,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    enable_prefix_caching=ENABLE_PREFIX_CACHING,  # IMPORTANT: Enables KV Cahce reuse between queries with shared prefixes
    max_logprobs=LOGPROBS + 20, # overhead to switch up and down
    trust_remote_code=True,
)

tokenizer = get_tokenizer(MODEL_NAME)

# OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_KEY"))

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.01s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:03<00:00,  3.01s/it]
(EngineCore pid=3300663) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 15.93it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 21.26it/s]


In [10]:
# Lists of allowed words
swedish_word_list = swedish_vocab["word"].tolist()
spanish_word_list = spanish_vocab["word"].tolist()
korean_word_list = korean_vocab["word"].tolist()

In [11]:
sampling_params = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    logprobs=LOGPROBS,
    repetition_penalty=REPETITION_PENALTY,
    # presence_penalty=PRESENCE_PENALTY,
    frequency_penalty=FREQUENCY_PENALTY,
)

EOS_TOKEN_ID = tokenizer.eos_token_id

swedish_lemmatizer = spacy.load("sv_core_news_sm")  # for Swedish (small)
# swedish_lemmatizer = spacy.load("sv_core_news_lg")  # for Swedish (large)
spanish_lemmatizer = spacy.load("es_core_news_sm")  # for Spanish (small)
# spanish_lemmatizer = spacy.load("es_dep_news_trf")  # for Spanish (large)
korean_lemmatizer = spacy.load("ko_core_news_sm")  # for Korean (small)
# korean_lemmatizer = spacy.load("ko_core_news_lg")  # for Korean (large)

In [12]:
system_prompt = "Du är en hjälpsam assistent. {vocab}"
system_prompt_special = """
Du är en hjälpsam assistent. Jag vill att du ska svara så naturligt som möjligt, men du har en regel: 
att alla tokens/ord du får använda måste finnas med i följande lista. Alla böjningar av orden i listan är tillåtna och räknas som samma ord (t.ex. säga, säger, sa, sagt). Skriv även så naturlig svenska som möjligt, även om det kräver böjningar eller små variationer av orden, dessutom Om det behövs för att skapa naturligt språk, får du använda ett fåtal vanliga småord utanför listan. Svara lika utförligt och naturligt som du normalt skulle göra utan begränsningen. Du är även helt tillåten att använda hur många namn som helst, oavsett om detta är namn på personer, städer, länder: ALLOWED WORD LIST:\n {vocab} \n END ALLOWED WORD LIST\n\n"""  # system prompt MUST use {vocab} to customly input correct vocab
user_prompt = "Vad är de tre bästa städerna som student i Sverige?"
# system_prompt = "You are a helpful assistant."
# user_prompt = "Tell me a short story about a guy and a ring."

In [13]:
from src.ai.models import Custom_vLLM, Vanilla_vLLM, Vanilla_ChatGPT

In [14]:
custom_vllm = Custom_vLLM(
    model=llm,
    tokenizer=tokenizer,
    lemmatizer=swedish_lemmatizer,
    allowed_words=swedish_word_list,
    beam_size=BEAM_SIZE,
    word_soft_constraint_penalty=WORD_SOFT_CONSTRAINT_PENALTY,
    alpha=LOGPROB_NORM_ALPHA
)

vanilla_vllm = Vanilla_vLLM(model=llm, tokenizer=tokenizer, allowed_words=swedish_word_list)
vanilla_chatgpt = Vanilla_ChatGPT(client=client, model="gpt-5.2", allowed_words=swedish_word_list)

In [53]:
custom_vllm(system_prompt, user_prompt, sampling_params, MAX_RESPONSE_TOKEN_LENGTH, VERBOSE, USE_WORD_CONSTRAINT, WORD_CONSTRAINT_TYPE)

Sequence step: 1
Sequence step: 2
Sequence step: 3
Sequence step: 4
Sequence step: 5
Sequence step: 6
Sequence step: 7
Sequence step: 8
Sequence step: 9
Sequence step: 10
Sequence step: 11
Sequence step: 12
Sequence step: 13
Sequence step: 14
Sequence step: 15
Sequence step: 16
Sequence step: 17
Sequence step: 18
Sequence step: 19
Sequence step: 20
Sequence step: 21
Sequence step: 22
Sequence step: 23
Sequence step: 24
Sequence step: 25
Sequence step: 26
Sequence step: 27
Sequence step: 28
Sequence step: 29
Sequence step: 30
Sequence step: 31
Sequence step: 32
Sequence step: 33
Sequence step: 34
Sequence step: 35
Sequence step: 36
Sequence step: 37
Sequence step: 38
Sequence step: 39
Sequence step: 40
Sequence step: 41
Sequence step: 42
Sequence step: 43
Sequence step: 44
Sequence step: 45
Sequence step: 46
Sequence step: 47
Sequence step: 48
Sequence step: 49
Sequence step: 50
Sequence step: 51
Sequence step: 52
Sequence step: 53
Sequence step: 54
Sequence step: 55
Sequence step: 56
S

'För att kunna ange de tre bästa städerna som student i Sverige, behöver jag kunna få mer information om dina preferenser och behov. Det är viktigt att säga att Sverige är en stor land med en stor variation av städer, och varje stad har sina egna förmågor och problem. Här är tre exempel på städer som är populära för studenter:\n\n1. **Umeå**: En av de största och mest studenterfyllda länderna i Sverige, med en stor universitetssjukhus och en bra infrastruktur för studenter.\n\n2. **Lund**: En av de mest studenterfyllda länderna i Sverige, med en stor universitetssjukhus och en bra infrastruktur för studenter. Det är också en av de länderna med en av de största universitetssjukhusen i Sverige.\n\n3. **Stockholm**: En av de största och mest interesserade städerna i Sverige, med en stor universitetssjukhus och en bra infrastruktur för studenter. Det är också en av de länderna med en av de största universitetssjukhusen i Sverige.\n\nDet är viktigt att se till att du kontrollerar att dessa 

In [ ]:
sampling_params_vanilla = SamplingParams(
    temperature=TEMPERATURE,
    max_tokens=200,  # supports full word generation
    logprobs=0,
    repetition_penalty=REPETITION_PENALTY,
    # presence_penalty=PRESENCE_PENALTY,
    frequency_penalty=FREQUENCY_PENALTY,
)

vanilla_vllm(
    system_prompt,
    user_prompt,
    sampling_params_vanilla,
    MAX_RESPONSE_TOKEN_LENGTH,
    prompt_allowed_words=True,
    verbose=True,
)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.80s/it, est. speed input: 219.65 toks/s, output: 110.93 toks/s]


In [41]:
vanilla_chatgpt_output, vanilla_chatgpt_perplexity = vanilla_chatgpt(
    system_prompt_special,
    user_prompt,
    MAX_RESPONSE_TOKEN_LENGTH,
    prompt_allowed_words=True,
)

In [ ]:
custom_vllm.beam_tree.visualize_tree("hayy_test_TEST")

In [21]:
nlp = spacy.load("sv_core_news_lg")

In [65]:
def calculate_lexical_percentage(chosen_final_answer):
    final_answer_list = re.sub(r"[^\w\s]", "", contractions.fix(chosen_final_answer))

    doc = nlp(final_answer_list)
    lemmas = [item.lemma_ for item in doc]
    pos = [item.pos_ for item in doc]

    excluded_classes = set(["X", "SYM", "PROPN", "NUM", "SPACE", "PUNCT"])

    exclude_indecies = [i for i, item in enumerate(pos) if item in excluded_classes]

    final_answer_list = [item for i, item in enumerate(lemmas) if i not in exclude_indecies]
    pos = [item for i, item in enumerate(pos) if i not in exclude_indecies]

    counter = sum(1 for x in final_answer_list if x in swedish_vocab["word"].tolist())
    total = len(final_answer_list)

    for x in final_answer_list:
        if x not in swedish_vocab["word"].tolist():
            print(x)

    return f"{round((counter/total) * 100, 2)}%"

In [66]:
custom_vllm_result = calculate_lexical_percentage(custom_vllm.output)
print(custom_vllm_result)

ange
stad
student
behöva
information
preferens
behov
viktig
variation
stad
varje
stad
egen
förmåga
problem
Häare
exempel
stad
populär
student
studenterfylld
universitetssjukhus
infrastruktur
student
studenterfylld
universitetssjukhus
infrastruktur
student
universitetssjukhuse
interesserad
stad
universitetssjukhus
infrastruktur
student
universitetssjukhuse
viktig
kontrollera
denna
studenterförmåga
behov
exempel
sjukhus
bibliotek
resurs
73.33%


In [67]:
vanilla_vllm_result = calculate_lexical_percentage(vanilla_vllm.output)
print(vanilla_vllm_result)

finnas
många
ländera
god
kvalitet
livsid
närhet
universitetslärande
områder
samt
utforma
infrastruktur
student
bryta
varje
genom
The
student
Room
StudentLife
engelsk
hitta
version
historiskaste
landshuvud
interaktiv
biblioteksscenarium
og
ofullständig
täckning
universiteit
ligga
stadsmarknad
gott
mat
pris
university
bygga
sats
you
are
flera
olik
uppklassa
kommunalbygdor
vilken
ännu
intressant
besök
tvekan
ni
överväcka
plats
snabbs
mög
var
försiktig
ihalsa
sp
35.16%


In [68]:
vanilla_chatgpt_result = calculate_lexical_percentage(vanilla_chatgpt.output)
print(vanilla_chatgpt_result)

stad
student
använda
ord
utanfö
lista
ord
listan
fler
ord
tex
namn
stad
lista
te
student
mm
75.36%


In [ ]:
# final_answer_list

['jag',
 'tycka',
 'att',
 'en',
 'bra',
 'stad',
 'som',
 'student',
 'i',
 'svensk',
 'land',
 'ofta',
 'vara',
 'Stor',
 'stad',
 'med',
 'mycket',
 'arbete',
 'kultur',
 'och',
 'många',
 'möjlighet',
 'men',
 'också',
 'väldig',
 'hög',
 'kostnad',
 'för',
 'bo',
 'och',
 'leva',
 'Stor',
 'stad',
 'vid',
 'vatten',
 'med',
 'universitet',
 'och',
 'bra',
 'student',
 'liv',
 'ganska',
 'mycket',
 'kultur',
 'och',
 'jobb',
 'och',
 'ofta',
 'lite',
 'lätt',
 'än',
 'stad',
 'med',
 'lång',
 'historia',
 'och',
 'stark',
 'student',
 'tradition',
 'med',
 'många',
 'student',
 'nation',
 'och',
 'mycket',
 'aktivitet',
 'inte',
 'lika',
 'stor',
 'som',
 'men',
 'ändå',
 'väldig',
 'bra',
 'som',
 'student',
 'vilken',
 'sak',
 'vara',
 'viktig',
 'för',
 'du',
 'kostnad',
 'jobb',
 'kultur',
 'eller',
 'student',
 'liv']

### Statistics Modeling

In statistics modeling it includes importing the created outputs and running the corrector to both get useful data for later evaluation, and prompting for iterative improvements

In [ ]:
from ai.OLD_models import Corrector
from src.ai.prompts import LLM_LEXICAL_SYSTEM_PROMPT

In [ ]:
corrector = Corrector() # Define corrector

# make flashcards into a list
list_of_flashcards = conversation.flashcards.split("\n")

# fake_history = [
#     {
#         "role": "user",
#         "content": [{"type": "output_text", "text": "Jag älskar pommes. Det måste finnas i mitt liv."}],
#     },
# ]

# corrector.fit(fake_history, list_of_flashcards)
corrector.fit(conversations_data, list_of_flashcards)

In [ ]:
display(Markdown(conversations_data[1].response))

In [ ]:
# Testing lexical constraints one time

## Lexical classification using separate LLM
llm_classified = corrector.lexical.llm_classification(client, "gpt-4o-mini", LLM_LEXICAL_SYSTEM_PROMPT)
# display(llm_classified.head(5)) # structure


llm_classification_ci_news = [[] for _ in range(MODEL_SAMPLE_SIZE)]
llm_classification_ci_olds = [[] for _ in range(MODEL_SAMPLE_SIZE)]

for obj in conversations_data:

    ci_old_ratio = 100 - round(obj.lexical.llm_classification["is_new"].to_numpy().sum()/len(obj.lexical.llm_classification["is_new"].tolist()) * 100)
    ci_new_ratio = round(
        obj.lexical.llm_classification["is_new"].to_numpy().sum()
        / len(obj.lexical.llm_classification["is_new"].tolist())
        * 100
    )

    print(f"Model CI ratio according to LLM (old/new): {ci_old_ratio}/{ci_new_ratio}")

    llm_classification_ci_news[obj.sample_id].append(ci_new_ratio)
    llm_classification_ci_olds[obj.sample_id].append(ci_old_ratio)


## Lexical classification using traditional NLP

In [ ]:
# test_corrector = Corrector()
# test_corrector.fit([
#     {
#         "role": "user",
#         "content": [{"type": "output_text", "text": "Jag älskar pommes. Det måste finnas i mitt liv."}],
#     },
# ], list_of_flashcards)


res = corrector.lexical.raw_checking()

raw_checking_ci_news = [[] for _ in range(MODEL_SAMPLE_SIZE)]
raw_checking_ci_olds = [[] for _ in range(MODEL_SAMPLE_SIZE)]

for obj in conversations_data:

    ci_old_ratio = 100 - round(obj.lexical.raw_checking["score"].to_numpy().sum()/len(obj.lexical.raw_checking["score"].tolist()) * 100)
    ci_new_ratio = round(
        obj.lexical.raw_checking["score"].to_numpy().sum()
        / len(obj.lexical.raw_checking["score"].tolist())
        * 100
    )

    print(f"Model CI ratio according to traditional NLP (old/new): {ci_old_ratio}/{ci_new_ratio}")

    raw_checking_ci_news[obj.sample_id].append(ci_new_ratio)
    raw_checking_ci_olds[obj.sample_id].append(ci_old_ratio)

In [ ]:
# Process CI ratios to put all samples together for calculating std etc
x = np.linspace(BATCH_SIZE, TOTAL_SIZE, TOTAL_SIZE // BATCH_SIZE)

y_llm_classification = np.array(llm_classification_ci_news)
llm_classification_means = np.mean(y_llm_classification, axis=0)
llm_classification_stds = np.std(y_llm_classification, axis=0)


y_raw_checking = np.array(raw_checking_ci_news)
raw_checking_means = np.mean(y_raw_checking, axis=0)
raw_checking_stds = np.std(y_raw_checking, axis=0)

In [ ]:
# Lineplot using

fig, axis = plt.subplots(1, 2, figsize=(20, 6))

axis[0].plot(
    x,
    llm_classification_means,
    label="(Mean) Words classified as NOT being from the Allowed Flashcards List",
    color="red",
)
axis[0].fill_between(
    x,
    llm_classification_means - llm_classification_stds,
    llm_classification_means + llm_classification_stds, alpha=0.3, label="Standard Deviation", color="orange")
axis[0].set_title(f"% of total tokens in output text classified BY AN LLM as being outside Allowed Flashcard List")

axis[1].plot(
    x,
    raw_checking_means,
    label="(Mean) Words classified as NOT being from the Allowed Flashcards List",
    color="red",
)
axis[1].fill_between(
    x,
    raw_checking_means - raw_checking_stds,
    raw_checking_means + raw_checking_stds,
    alpha=0.3,
    label="Standard Deviation",
    color="orange"
)
axis[1].set_title(
    f"% of total tokens in output text classified BY TRADITIONAL NLP as being outside Allowed Flashcard List"
)

for ax in axis:
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    ax.set_ylim(0, 100)
    ax.axhline(y=10, linestyle="--", label="Maximum Allowed Threshold")
    ax.set_xlabel("Nr. Unique Allowed Flashcards")
    ax.set_ylabel("Classification (%)")
    ax.legend()

plt.show()

### Improvement Loop

Improvement loop loops between model and corrector to improve the final output to later measure the improvement and total performance in real-world tasks of this subsystem.

### Evaluation